# Synthetic Covariate Shift SCRC Experiment

**Research question**: Under *pure* covariate shift, does GNN-DRE's higher ESS (~31%) translate
into honest FNR ≤ α guarantees on the target, while LR-DRE fails due to low ESS?

## Setup
CheXpert is randomly split 60/40 into **Source** (clean) and **Target** (Gaussian-blurred):
- **No label shift**: P(Y) identical by construction (same dataset, random split)
- **No concept shift**: P(Y|X) preserved (disease concepts unchanged by blur)
- **Pure covariate shift**: P(X) differs due to image degradation

Under these conditions, conformal theory predicts GNN Test FNR ≈ α when ESS is high.

## Architecture
```
CheXpert (N=64,534)
├── Source (60%) — CLEAN pre-extracted features
│   ├── Train  (75% of Source)  → fit LR + GNN
│   └── Cal    (25% of Source)  → SCRC calibration
└── Target (40%) — PERTURBED features (re-extracted with Gaussian blur)
    ├── DRE Pool (50% of Target) → fit DRE domain classifier
    └── Test     (50% of Target) → SCRC evaluation
```

## Three SCRC arms
| Arm | Probs | DRE | Expected ESS |
|-----|-------|-----|--------------|
| LR-DRE (nc)  | LR predict_proba | 1024-dim PCA-4, no clip | ~1% |
| LR-DRE (c)   | LR predict_proba | 1024-dim PCA-4, clip=20 | ~6% |
| GNN-DRE (c)  | GNN sigmoid      | 7-dim prob space, clip=20 | ~31% |

In [ ]:
from pathlib import Path
import torch

ROOT = Path('/Users/amo/programData/wcp-l2d')

# ============================================================
# USER CONFIGURATION — set CHEXPERT_IMGPATH before running
# Goldilocks tuning (Cell 5) and feature extraction (Cell 6)
# ============================================================
CHEXPERT_IMGPATH = '/Users/amo/programData/wcp-l2d/data/chexpert'
# ============================================================

SIGMA   = 3.0    # Gaussian blur sigma (tune in Cell 5 or set manually)
SEED    = 42
BETA    = 0.15   # Stage 1 deferral budget
ALPHA   = 0.10   # Stage 2 FNR target
K       = 7
DEVICE  = 'mps' if torch.backends.mps.is_available() else 'cpu'

FEAT_DIR        = ROOT / 'data' / 'features'
PERTURBED_CACHE = FEAT_DIR / f'chexpert_target_perturbed_sigma{SIGMA:.1f}_features.npz'

print(f'Device:           {DEVICE}')
print(f'SIGMA:            {SIGMA}')
print(f'BETA / ALPHA:     {BETA} / {ALPHA}')
print(f'PERTURBED_CACHE:  {PERTURBED_CACHE}')
if CHEXPERT_IMGPATH is None:
    print('\nNOTE: CHEXPERT_IMGPATH is None.')
    print('  Cells 5-6 will be skipped if perturbed cache does not exist.')
    print(f'  Cache exists: {PERTURBED_CACHE.exists()}')

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from torchvision import transforms
from torch.utils.data import Subset as TorchSubset

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from wcp_l2d.dre import AdaptiveDRE
from wcp_l2d.features import ExtractedFeatures, load_model, extract_features
from wcp_l2d.data import load_and_align_dataset, apply_xrv_transforms, make_dataloader
from wcp_l2d.gnn import build_adjacency_matrix, train_gnn
from wcp_l2d.pathologies import COMMON_PATHOLOGIES
from wcp_l2d.scrc import (
    multilabel_entropy, select_for_deferral, calibrate_per_pathology_crc_fnr
)

np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({
    'figure.dpi': 100, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
})

print(f'Pathologies: {COMMON_PATHOLOGIES}')


class GaussianBlurNP:
    """Gaussian blur on numpy (C, H, W) arrays. Compatible with xrv transforms."""
    def __init__(self, sigma):
        self.sigma = sigma
    def __call__(self, img):
        return gaussian_filter(img, sigma=[0, self.sigma, self.sigma])

## 1. Load Pre-extracted Features + 60/40 Index Split

- **Source** (60%): clean, pre-extracted features — used as-is
- **Target** (40%): same CheXpert images but will be re-extracted with Gaussian blur
- Labels are identical for both halves → **no label shift by construction**

In [ ]:
chex = ExtractedFeatures.load(FEAT_DIR / 'chexpert_densenet121-res224-chex_features.npz')
print(f'CheXpert: {chex.features.shape}  labels: {chex.labels.shape}')

rng        = np.random.RandomState(SEED)
all_pos    = rng.permutation(len(chex.features))
n_source   = int(0.60 * len(chex.features))
source_pos = all_pos[:n_source]   # positions in .npz array → clean features
target_pos = all_pos[n_source:]   # positions in .npz array → will be perturbed

orig_target_idx = chex.indices[target_pos]   # original CheXpert dataset indices

X_source_raw = chex.features[source_pos]   # (n_source, 1024) — clean
Y_source     = chex.labels[source_pos]     # (n_source, 7)
Y_target     = chex.labels[target_pos]     # (n_target, 7) — same labels, NO label shift

print(f'Source: {X_source_raw.shape}  (60% clean pre-extracted)')
print(f'Target: {len(Y_target):,} samples  (40% — will use perturbed features)')
print(f'\nLabel shift verification (source vs target prevalence):')
print(f'{"Pathology":15s} | {"Source":8s} | {"Target":8s} | {"Delta":8s}')
print('-' * 50)
for k, name in enumerate(COMMON_PATHOLOGIES):
    sp = np.nanmean(Y_source[:, k])
    tp = np.nanmean(Y_target[:, k])
    print(f'{name:15s} | {sp:8.4f} | {tp:8.4f} | {tp-sp:+8.4f}')
print('\nExpected: all Delta ≈ 0 (same dataset, random split)')

## 2. Goldilocks Sigma Tuning (optional — requires CHEXPERT_IMGPATH)

Test σ ∈ {1.0, 2.0, 3.0, 5.0, 7.0} on a small subset (~500 images).

**Goldilocks criteria**:
- Domain AUC ∈ [0.90, 0.98]: shift is detectable but not extreme
- Mean AUC drop ∈ [0.05, 0.10]: meaningful but not catastrophic degradation

Skip this cell and use `SIGMA=3.0` if images are not available.

In [ ]:
# Goldilocks tuning skipped — SIGMA=3.0 pre-chosen.
# To run full tuning, replace this cell with the commented-out version in the source notebook.
print(f"Goldilocks tuning skipped.  Using pre-set SIGMA={SIGMA}.")


## 3. Extract Perturbed Target Features (cached)

Re-extract features for the 40% target subset with Gaussian blur applied.
Cached to `PERTURBED_CACHE` so this only runs once per sigma.

In [ ]:
if PERTURBED_CACHE.exists():
    print(f'Loading cached perturbed features: {PERTURBED_CACHE}')
    _cache       = np.load(PERTURBED_CACHE, allow_pickle=True)
    X_target_raw = _cache['features']
    assert _cache['labels'].shape == Y_target.shape, 'Label shape mismatch in cache!'
    print(f'Loaded X_target_raw: {X_target_raw.shape}')
    print('Label shape check: PASS')
else:
    if CHEXPERT_IMGPATH is None:
        raise RuntimeError(
            f'Cache {PERTURBED_CACHE} not found and CHEXPERT_IMGPATH is None.\n'
            'Set CHEXPERT_IMGPATH in Cell 1, run Goldilocks tuning, then retry.'
        )
    print(f'Extracting perturbed features for {len(orig_target_idx):,} images '
          f'(sigma={SIGMA}) ...')
    chex_ds = load_and_align_dataset('chexpert', CHEXPERT_IMGPATH)
    apply_xrv_transforms(chex_ds)
    chex_ds.transform = transforms.Compose([chex_ds.transform, GaussianBlurNP(SIGMA)])

    tgt_subset = TorchSubset(chex_ds, orig_target_idx.tolist())
    loader     = make_dataloader(tgt_subset, batch_size=128, num_workers=4)
    model_full = load_model(weights='densenet121-res224-chex', device=DEVICE)
    X_target_raw, _, _ = extract_features(model_full, loader, DEVICE)

    np.savez_compressed(
        PERTURBED_CACHE,
        features=X_target_raw,
        labels=Y_target,
        indices=orig_target_idx,
        pathologies=np.array(COMMON_PATHOLOGIES),
        dataset_name=np.array('chexpert_perturbed'),
        model_weights=np.array('densenet121-res224-chex'),
    )
    print(f'Saved to {PERTURBED_CACHE}')

print(f'X_target_raw shape: {X_target_raw.shape}')

## 4. Goldilocks Verification on Full Dataset

In [ ]:
# Domain AUC (PCA-4) on subsampled source vs full target
_rng_v      = np.random.RandomState(SEED)
_n_sample   = min(5000, len(X_source_raw), len(X_target_raw))
_src_idx    = _rng_v.choice(len(X_source_raw), _n_sample, replace=False)
_tgt_idx    = _rng_v.choice(len(X_target_raw), _n_sample, replace=False)
_X_dom      = np.vstack([X_source_raw[_src_idx], X_target_raw[_tgt_idx]])
_y_dom      = np.array([0] * _n_sample + [1] * _n_sample)
_sc_v       = StandardScaler().fit(_X_dom)
_pca_v      = PCA(n_components=4, random_state=SEED).fit(_sc_v.transform(_X_dom))
_X_pca      = _pca_v.transform(_sc_v.transform(_X_dom))
_lr_v       = LogisticRegression(solver='lbfgs', max_iter=500, random_state=SEED)
_lr_v.fit(_X_pca, _y_dom)
full_domain_auc = roc_auc_score(_y_dom, _lr_v.predict_proba(_X_pca)[:, 1])

print(f'Full-dataset Domain AUC (PCA-4, n={_n_sample}): {full_domain_auc:.4f}')
status = 'PASS' if 0.90 <= full_domain_auc <= 0.98 else 'OUTSIDE range — consider adjusting SIGMA'
print(f'Goldilocks target [0.90, 0.98]: {status}')

## 5. Sub-splits + StandardScaler

```
Source (60%)
  └── Train (75%) + Cal (25%)
Target (40%)
  └── DRE Pool (50%) + Test (50%)
```

StandardScaler fit on SOURCE train, applied to all sets.

In [ ]:
n_tr        = int(0.75 * n_source)
X_train_raw = X_source_raw[:n_tr]
Y_train     = Y_source[:n_tr]
X_cal_raw   = X_source_raw[n_tr:]
Y_cal       = Y_source[n_tr:]

n_pool      = len(X_target_raw) // 2
X_pool_raw  = X_target_raw[:n_pool]
X_test_raw  = X_target_raw[n_pool:]
Y_pool_tgt  = Y_target[:n_pool]
Y_test      = Y_target[n_pool:]

scaler  = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_cal   = scaler.transform(X_cal_raw)
X_pool  = scaler.transform(X_pool_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Train:  {X_train.shape}  (source clean)')
print(f'Cal:    {X_cal.shape}    (source clean)')
print(f'Pool:   {X_pool.shape}   (target perturbed, DRE fitting)')
print(f'Test:   {X_test.shape}   (target perturbed, SCRC evaluation)')
print(f'\nPure covariate shift check (prevalences should match):')
print(f'{"Pathology":15s} | {"Cal (src)":10s} | {"Test (tgt)":10s}')
print('-' * 42)
for k, name in enumerate(COMMON_PATHOLOGIES):
    print(f'{name:15s} | {np.nanmean(Y_cal[:, k]):10.4f} | {np.nanmean(Y_test[:, k]):10.4f}')

In [ ]:
# Label distribution per split: n_pos / n_neg / n_nan and their percentages
splits_info = [
    ('Train',    Y_train),
    ('Cal',      Y_cal),
    ('Pool',     Y_pool_tgt),
    ('Test',     Y_test),
]

for split_name, Y in splits_info:
    n = len(Y)
    print(f'{split_name}  (n={n:,}):')
    print(f'  {"Pathology":15s} | {"Pos n (%)":>14s} | {"Neg n (%)":>14s} | {"NaN n (%)":>14s}')
    print('  ' + '-' * 65)
    for k, name in enumerate(COMMON_PATHOLOGIES):
        n_pos = int((Y[:, k] == 1).sum())
        n_neg = int((Y[:, k] == 0).sum())
        n_nan = int(np.isnan(Y[:, k]).sum())
        print(
            f'  {name:15s} | {n_pos:5d} ({n_pos/n*100:5.1f}%) | '
            f'{n_neg:5d} ({n_neg/n*100:5.1f}%) | '
            f'{n_nan:5d} ({n_nan/n*100:5.1f}%)'
        )
    print()

## 6. Binary LR Classifiers (per-pathology, Source train)

Used for:
1. LR SCRC arm probabilities (`predict_proba`)
2. GNN residual logit initialisation (`decision_function`)

In [ ]:
lrs = []
for k, name in enumerate(COMMON_PATHOLOGIES):
    valid = ~np.isnan(Y_train[:, k])
    if valid.sum() < 10 or len(np.unique(Y_train[valid, k])) < 2:
        lrs.append(None)
        continue
    lr = LogisticRegression(solver='lbfgs', max_iter=500, random_state=SEED)
    lr.fit(X_train[valid], Y_train[valid, k].astype(int))
    lrs.append(lr)


def get_logits_lr(lrs_, X_s):
    """[N, K] decision_function for GNN residual init."""
    out = np.zeros((len(X_s), K), dtype=np.float32)
    for k, lr in enumerate(lrs_):
        if lr is not None:
            out[:, k] = lr.decision_function(X_s)
    return out


def get_proba_lr(lrs_, X_s):
    """[N, K] predict_proba[:, 1] for LR SCRC arm."""
    out = np.zeros((len(X_s), K), dtype=np.float32)
    for k, lr in enumerate(lrs_):
        if lr is not None:
            out[:, k] = lr.predict_proba(X_s)[:, 1]
    return out


init_tr   = get_logits_lr(lrs, X_train)
init_cal  = get_logits_lr(lrs, X_cal)
init_pool = get_logits_lr(lrs, X_pool)
init_test = get_logits_lr(lrs, X_test)

p_cal_lr  = get_proba_lr(lrs, X_cal)     # LR probs on SOURCE cal (for calibration)
p_test_lr = get_proba_lr(lrs, X_test)    # LR probs on TARGET test (for evaluation)

print('LR classifiers trained.')
print(f'p_cal_lr:  {p_cal_lr.shape}')
print(f'p_test_lr: {p_test_lr.shape}')

## 7. Label Co-occurrence Adjacency + GNN Training

In [ ]:
A = build_adjacency_matrix(Y_train, tau=0.10)
assert torch.allclose(A.sum(dim=1), torch.ones(K), atol=1e-5), 'Row sums must equal 1'
print(f'Adjacency: {A.shape}  non-zero off-diag: {int((A > 0).sum()) - K}/{K*(K-1)}')

print(f'\nTraining LabelGCN on {DEVICE} (50 epochs) ...')
gnn, history = train_gnn(
    features_train=X_train,
    labels_train=Y_train,
    features_val=X_cal,
    labels_val=Y_cal,
    adjacency=A,
    init_logits_train=init_tr,
    init_logits_val=init_cal,
    epochs=50,
    save_best=True,
    batch_size=512,
    lr=1e-3,
    weight_decay=1e-4,
    device=DEVICE,
    verbose=False,
)
best_ep = history['best_epoch'][0]
print(f'Best val AUC: {max(history["val_auc"]):.4f}  at epoch {best_ep}/50')

## 8. GNN Probability Extraction

In [ ]:
def get_probs_gnn(model, X_s, init_np, batch_size=2048):
    """Batched GNN forward pass → sigmoid [N, K]."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for start in range(0, len(X_s), batch_size):
            end  = min(start + batch_size, len(X_s))
            Xt   = torch.tensor(X_s[start:end], dtype=torch.float32)
            it   = torch.tensor(init_np[start:end], dtype=torch.float32)
            logi = model(Xt, it).numpy()
            all_probs.append(1.0 / (1.0 + np.exp(-logi)))
    return np.vstack(all_probs)


p_cal_gnn  = get_probs_gnn(gnn, X_cal,  init_cal)
p_pool_gnn = get_probs_gnn(gnn, X_pool, init_pool)
p_test_gnn = get_probs_gnn(gnn, X_test, init_test)

print(f'p_cal_gnn:  {p_cal_gnn.shape}')
print(f'p_pool_gnn: {p_pool_gnn.shape}')
print(f'p_test_gnn: {p_test_gnn.shape}')

# Per-pathology AUC on TARGET test: LR vs GNN
print(f'\nTarget test AUC (LR vs GNN):')
print(f'{"Pathology":15s} | {"LR AUC":8s} | {"GNN AUC":8s} | {"ΔAUC":8s}')
print('-' * 52)
for k, name in enumerate(COMMON_PATHOLOGIES):
    valid = ~np.isnan(Y_test[:, k])
    if valid.sum() < 2 or len(np.unique(Y_test[valid, k])) < 2:
        print(f'{name:15s} | {"N/A":8s} | {"N/A":8s}')
        continue
    auc_lr  = roc_auc_score(Y_test[valid, k], p_test_lr[valid, k])
    auc_gnn = roc_auc_score(Y_test[valid, k], p_test_gnn[valid, k])
    print(f'{name:15s} | {auc_lr:8.4f} | {auc_gnn:8.4f} | {auc_gnn-auc_lr:+8.4f}')

## 9. Density Ratio Estimation (three variants)

Source = SOURCE cal features  
Target = TARGET DRE pool features  

| Variant | Space | PCA | Clip | Expected ESS |
|---------|-------|-----|------|--------------|
| LR-DRE (nc) | 1024-dim raw | PCA-4 | None | ~1% |
| LR-DRE (c)  | 1024-dim raw | PCA-4 | 20.0 | ~6% |
| GNN-DRE (c) | 7-dim probs  | None  | 20.0 | ~31% |

In [ ]:
# 9a — LR-DRE no clip
dre_lr_nc = AdaptiveDRE(n_components=4, weight_clip=None, random_state=SEED)
dre_lr_nc.fit(source_features=X_cal_raw, target_features=X_pool_raw)
w_cal_lr_nc  = dre_lr_nc.compute_weights(X_cal_raw)
w_test_lr_nc = dre_lr_nc.compute_weights(X_test_raw)
diag_lr_nc   = dre_lr_nc.diagnostics(X_cal_raw)
print(f'LR-DRE (nc): domain_AUC={diag_lr_nc.domain_auc:.4f}, '
      f'ESS%={diag_lr_nc.ess_fraction*100:.2f}%, w_max={diag_lr_nc.weight_max:.1f}')

In [ ]:
# 9b — LR-DRE clip=20
dre_lr = AdaptiveDRE(n_components=4, weight_clip=20.0, random_state=SEED)
dre_lr.fit(source_features=X_cal_raw, target_features=X_pool_raw)
w_cal_lr  = dre_lr.compute_weights(X_cal_raw)
w_test_lr = dre_lr.compute_weights(X_test_raw)
diag_lr   = dre_lr.diagnostics(X_cal_raw)
print(f'LR-DRE (c):  domain_AUC={diag_lr.domain_auc:.4f}, '
      f'ESS%={diag_lr.ess_fraction*100:.2f}%, w_max={diag_lr.weight_max:.1f}')

In [ ]:
# 9c — GNN-DRE clip=20 (7-dim probability space)
dre_gnn = AdaptiveDRE(n_components=None, weight_clip=20.0, random_state=SEED)
dre_gnn.fit(source_features=p_cal_gnn, target_features=p_pool_gnn)
w_cal_gnn  = dre_gnn.compute_weights(p_cal_gnn)
w_test_gnn = dre_gnn.compute_weights(p_test_gnn)
diag_gnn   = dre_gnn.diagnostics(p_cal_gnn)
print(f'GNN-DRE (c): domain_AUC={diag_gnn.domain_auc:.4f}, '
      f'ESS%={diag_gnn.ess_fraction*100:.2f}%, w_max={diag_gnn.weight_max:.1f}')

In [ ]:
dre_table = pd.DataFrame([
    {'Method': 'LR-DRE (no clip)', 'Domain AUC': round(diag_lr_nc.domain_auc, 4),
     'ESS%': round(diag_lr_nc.ess_fraction * 100, 2),
     'W_mean': round(float(w_cal_lr_nc.mean()), 3), 'W_max': round(float(w_cal_lr_nc.max()), 1)},
    {'Method': 'LR-DRE (clip=20)', 'Domain AUC': round(diag_lr.domain_auc, 4),
     'ESS%': round(diag_lr.ess_fraction * 100, 2),
     'W_mean': round(float(w_cal_lr.mean()), 3),    'W_max': round(float(w_cal_lr.max()), 1)},
    {'Method': 'GNN-DRE (clip=20)', 'Domain AUC': round(diag_gnn.domain_auc, 4),
     'ESS%': round(diag_gnn.ess_fraction * 100, 2),
     'W_mean': round(float(w_cal_gnn.mean()), 3),   'W_max': round(float(w_cal_gnn.max()), 1)},
])
print('DRE Weight Quality Comparison')
print(dre_table.to_string(index=False))
print('\nExpected: ESS%(GNN) >> ESS%(LR) under pure covariate shift')

## 10. Stage 1 — Global Entropy Deferral (β = 0.15)

Defers top-β fraction by multi-label entropy from GNN probs.
Stage 1 deferral mask is **shared** across all three SCRC arms.

In [ ]:
entropy_cal = multilabel_entropy(p_cal_gnn)
entropy_tst = multilabel_entropy(p_test_gnn)
defer_cal   = select_for_deferral(entropy_cal, BETA, seed=SEED)
defer_tst   = select_for_deferral(entropy_tst, BETA, seed=SEED)

print(f'Stage 1 (β={BETA}):')
print(f'  Cal:  {defer_cal.sum():,}/{len(defer_cal):,} deferred  ({defer_cal.mean()*100:.1f}%)')
print(f'  Test: {defer_tst.sum():,}/{len(defer_tst):,} deferred  ({defer_tst.mean()*100:.1f}%)')
assert defer_cal.mean() <= BETA + 1e-9
assert defer_tst.mean() <= BETA + 1e-9
print('Deferral rate budget check: PASS')

## 11. Stage 2 — Per-pathology SCRC Calibration (3 arms)

For each arm, calibrate λ_k* = sup{λ : weighted_FNR_k(λ) ≤ α_k} on SOURCE cal.

- **Arm 1**: LR probs + LR-DRE (no clip)  
- **Arm 2**: LR probs + LR-DRE (clip=20)  
- **Arm 3**: GNN probs + GNN-DRE (clip=20)

In [ ]:
alpha_arr = np.full(K, ALPHA)

# Arm 1: LR probs + LR-DRE no clip
crc_lr_nc = calibrate_per_pathology_crc_fnr(
    probs=p_cal_lr[~defer_cal], labels=Y_cal[~defer_cal],
    weights=w_cal_lr_nc[~defer_cal], alpha=alpha_arr, n_grid=1001,
    pathology_names=COMMON_PATHOLOGIES,
)

# Arm 2: LR probs + LR-DRE clip=20
crc_lr = calibrate_per_pathology_crc_fnr(
    probs=p_cal_lr[~defer_cal], labels=Y_cal[~defer_cal],
    weights=w_cal_lr[~defer_cal], alpha=alpha_arr, n_grid=1001,
    pathology_names=COMMON_PATHOLOGIES,
)

# Arm 3: GNN probs + GNN-DRE clip=20
crc_gnn = calibrate_per_pathology_crc_fnr(
    probs=p_cal_gnn[~defer_cal], labels=Y_cal[~defer_cal],
    weights=w_cal_gnn[~defer_cal], alpha=alpha_arr, n_grid=1001,
    pathology_names=COMMON_PATHOLOGIES,
)

print(f'Stage 2 Calibration Results (all Cal FNR should be ≤ α={ALPHA}):')
print(f'{"Pathology":15s} | {"LR-nc λ*":8s} {"FNR":5s} | {"LR-c λ*":8s} {"FNR":5s} | {"GNN λ*":7s} {"FNR":5s}')
print('-' * 72)
for k, name in enumerate(COMMON_PATHOLOGIES):
    print(
        f'{name:15s} | {crc_lr_nc.lambda_hats[k]:.3f}    {crc_lr_nc.weighted_fnr_at_lambda[k]:.3f} | '
        f'{crc_lr.lambda_hats[k]:.3f}    {crc_lr.weighted_fnr_at_lambda[k]:.3f} | '
        f'{crc_gnn.lambda_hats[k]:.3f}   {crc_gnn.weighted_fnr_at_lambda[k]:.3f}'
    )
print('-' * 72)
print(
    f'{"Mean":15s} | {crc_lr_nc.lambda_hats.mean():.3f}    '
    f'{crc_lr_nc.weighted_fnr_at_lambda.mean():.3f} | '
    f'{crc_lr.lambda_hats.mean():.3f}    '
    f'{crc_lr.weighted_fnr_at_lambda.mean():.3f} | '
    f'{crc_gnn.lambda_hats.mean():.3f}   '
    f'{crc_gnn.weighted_fnr_at_lambda.mean():.3f}'
)

# Sanity: all cal FNRs ≤ alpha
for arm_name, crc in [('LR-nc', crc_lr_nc), ('LR-c', crc_lr), ('GNN-c', crc_gnn)]:
    ok = (crc.weighted_fnr_at_lambda <= ALPHA + 1e-9).all()
    print(f'Cal FNR ≤ {ALPHA} [{arm_name}]: {"PASS" if ok else "FAIL"}')

## 12. Test Evaluation + Core Hypothesis Check

**Core hypothesis**: Under pure covariate shift, GNN-DRE (high ESS) should give
Test FNR ≈ α = 0.10, while LR-DRE (low ESS) diverges from α.

**FNR Gap** = |Test FNR − α| — lower is better (tighter theoretical guarantee).

In [ ]:
def evaluate_fnr_fpr(probs_kept, labels_kept, lambda_stars):
    """Empirical FNR and FPR per pathology on already-filtered (non-deferred) samples."""
    K_ = probs_kept.shape[1]
    fnrs, fprs = np.zeros(K_), np.zeros(K_)
    for k in range(K_):
        valid = ~np.isnan(labels_kept[:, k])
        yv    = labels_kept[valid, k]
        pv    = probs_kept[valid, k]
        pred  = (pv >= lambda_stars[k]).astype(float)
        pos   = yv == 1
        neg   = yv == 0
        fnrs[k] = (pred[pos] == 0).mean() if pos.sum() > 0 else float('nan')
        fprs[k] = (pred[neg] == 1).mean() if neg.sum() > 0 else float('nan')
    return fnrs, fprs


kept_tst = ~defer_tst

fnr_lr_nc, fpr_lr_nc = evaluate_fnr_fpr(
    p_test_lr[kept_tst],  Y_test[kept_tst], crc_lr_nc.lambda_hats)
fnr_lr,    fpr_lr    = evaluate_fnr_fpr(
    p_test_lr[kept_tst],  Y_test[kept_tst], crc_lr.lambda_hats)
fnr_gnn,   fpr_gnn   = evaluate_fnr_fpr(
    p_test_gnn[kept_tst], Y_test[kept_tst], crc_gnn.lambda_hats)

print('=' * 95)
print('CORE HYPOTHESIS: Under pure covariate shift, GNN-DRE Test FNR ≈ α = 0.10')
print('  (high ESS → reliable importance weights → calibration guarantee transfers)')
print('=' * 95)

print(f'\n{"Method":22s} | {"ESS%":5s} | {"Cal FNR":8s} | {"Test FNR":9s} | {"FNR Gap":8s} | {"Test FPR":8s}')
print('-' * 75)
methods_summary = [
    ('LR-DRE (nc)',
     diag_lr_nc.ess_fraction * 100,
     np.nanmean(crc_lr_nc.weighted_fnr_at_lambda),
     np.nanmean(fnr_lr_nc), np.nanmean(fpr_lr_nc)),
    ('LR-DRE (clip=20)',
     diag_lr.ess_fraction * 100,
     np.nanmean(crc_lr.weighted_fnr_at_lambda),
     np.nanmean(fnr_lr), np.nanmean(fpr_lr)),
    ('GNN-DRE (clip=20)',
     diag_gnn.ess_fraction * 100,
     np.nanmean(crc_gnn.weighted_fnr_at_lambda),
     np.nanmean(fnr_gnn), np.nanmean(fpr_gnn)),
]
for name, ess, cal_fnr, test_fnr, test_fpr in methods_summary:
    gap = abs(test_fnr - ALPHA)
    print(f'{name:22s} | {ess:5.1f} | {cal_fnr:8.3f} | {test_fnr:9.3f} | {gap:8.3f} | {test_fpr:8.3f}')

print(f'\nPer-pathology detail:')
print(f'{"Pathology":15s} | {"LR-nc":14s} | {"LR-c":14s} | {"GNN-c":14s}')
print(f'{"":15s} | {"FNR":6s} {"FPR":6s} | {"FNR":6s} {"FPR":6s} | {"FNR":6s} {"FPR":6s}')
print('-' * 72)
for k, name in enumerate(COMMON_PATHOLOGIES):
    print(
        f'{name:15s} | {fnr_lr_nc[k]:.3f}  {fpr_lr_nc[k]:.3f} | '
        f'{fnr_lr[k]:.3f}  {fpr_lr[k]:.3f} | '
        f'{fnr_gnn[k]:.3f}  {fpr_gnn[k]:.3f}'
    )
print('-' * 72)
print(
    f'{"Mean":15s} | {np.nanmean(fnr_lr_nc):.3f}  {np.nanmean(fpr_lr_nc):.3f} | '
    f'{np.nanmean(fnr_lr):.3f}  {np.nanmean(fpr_lr):.3f} | '
    f'{np.nanmean(fnr_gnn):.3f}  {np.nanmean(fpr_gnn):.3f}'
)

## 13. Visualization

In [ ]:
(ROOT / 'report').mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 4, figsize=(26, 6))
x        = np.arange(K)
width    = 0.26
colors   = ['#1f77b4', '#ff7f0e', '#2ca02c']
offsets  = [-1, 0, 1]
methods_plot = [
    ('LR-DRE (nc)',    fnr_lr_nc, fpr_lr_nc,  colors[0]),
    ('LR-DRE (c=20)',  fnr_lr,    fpr_lr,     colors[1]),
    ('GNN-DRE (c=20)', fnr_gnn,   fpr_gnn,    colors[2]),
]

# Panel 1: FNR per pathology
ax = axes[0]
for (lbl, fnrs, _, col), off in zip(methods_plot, offsets):
    ax.bar(x + off * width, fnrs, width, label=lbl, color=col, alpha=0.85)
ax.axhline(ALPHA, color='red', ls='--', lw=1.8, label=f'α={ALPHA}')
ax.set_xticks(x)
ax.set_xticklabels(COMMON_PATHOLOGIES, rotation=40, ha='right', fontsize=9)
ax.set_title('FNR per Pathology\n(dashed = α target)', fontsize=11)
ax.set_ylabel('False Negative Rate')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

# Panel 2: FPR per pathology
ax = axes[1]
for (lbl, _, fprs, col), off in zip(methods_plot, offsets):
    ax.bar(x + off * width, fprs, width, label=lbl, color=col, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(COMMON_PATHOLOGIES, rotation=40, ha='right', fontsize=9)
ax.set_title('FPR per Pathology\n(lower = fewer false alarms)', fontsize=11)
ax.set_ylabel('False Positive Rate')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

# Panel 3: FNR Gap = |Test FNR - alpha|
ax = axes[2]
method_names = ['LR-DRE\n(nc)', 'LR-DRE\n(c=20)', 'GNN-DRE\n(c=20)']
fnr_gaps = [
    abs(np.nanmean(fnr_lr_nc) - ALPHA),
    abs(np.nanmean(fnr_lr)    - ALPHA),
    abs(np.nanmean(fnr_gnn)   - ALPHA),
]
bars = ax.bar(method_names, fnr_gaps, color=colors, alpha=0.85)
for bar, val in zip(bars, fnr_gaps):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(0, color='black', lw=0.5)
ax.set_title('Mean |FNR Gap| = |Test FNR − α|\n(lower → tighter guarantee)', fontsize=11)
ax.set_ylabel('|Test FNR − α|')

# Panel 4: ESS comparison
ax = axes[3]
ess_vals = [
    diag_lr_nc.ess_fraction * 100,
    diag_lr.ess_fraction    * 100,
    diag_gnn.ess_fraction   * 100,
]
ess_bars = ax.bar(method_names, ess_vals, color=colors, alpha=0.85)
for bar, val in zip(ess_bars, ess_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Effective Sample Size (ESS%)\n(higher → more reliable weights)', fontsize=11)
ax.set_ylabel('ESS %')

plt.suptitle(
    f'Synthetic Pure Covariate Shift SCRC (σ={SIGMA}, β={BETA}, α={ALPHA})\n'
    'CheXpert 60/40 split: Source (clean) → Target (Gaussian-blurred)',
    fontsize=13, fontweight='bold', y=1.02,
)
plt.tight_layout()
fig_path = ROOT / 'report' / f'synthetic_covariate_shift_scrc_sigma{SIGMA:.1f}.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 14. Summary & Interpretation

### Key predictions (if theory holds under pure covariate shift)

| Metric | LR-DRE (nc) | LR-DRE (c) | GNN-DRE (c) | Verdict |
|--------|------------|-----------|------------|--------|
| ESS% | ~1% | ~6% | ~31% | GNN >> LR |
| Cal FNR | ≤ 0.10 | ≤ 0.10 | ≤ 0.10 | All pass |
| Test FNR | >> 0.10 | > 0.10 | ≈ 0.10 | **GNN honest** |
| FNR Gap | large | medium | ≈ 0 | **GNN wins** |

### Why this matters
Under pure covariate shift (no label/concept shift), the importance-weighted
conformal guarantee is exact when:
1. The density ratio w(x) = p_target(x)/p_source(x) is well-estimated
2. ESS is high enough to trust the weighted empirical quantile

GNN-DRE operates in the 7-dim probability space where domain separability is
lower (AUC ≈ 0.84 vs 0.97 for LR), leading to better-calibrated weights and
higher ESS (~31% vs ~6%). This validates the core claim of the paper:
> *Representation-aware DRE via GNN enables honest FNR guarantees under
> distribution shift in medical imaging.*